# 07 – Truy xuất và đánh giá


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

INDEX_PATH = PROJECT_ROOT / "knowledge" / "index" / "faiss.index"
META_PATH = PROJECT_ROOT / "knowledge" / "index" / "chunks_meta.json"
CHUNKS_PATH = PROJECT_ROOT / "knowledge" / "chunks" / "chunks.jsonl"
VACCINES_PATH = PROJECT_ROOT / "data" / "mock" / "vaccines_from_db.json"

print("index exists", INDEX_PATH.exists())

index exists True


In [2]:
import json
from app.chatbot.rag.embedder import load_embedder
from app.chatbot.rag.vector_store import VectorStore
from app.chatbot.rag.retriever import Retriever

metas = json.loads(META_PATH.read_text(encoding="utf-8"))
texts = [m.get("text") or "" for m in metas]

embedder = load_embedder(prefer_bge=True, corpus_for_tfidf=texts)
store = VectorStore.load(INDEX_PATH, META_PATH)
print("store", store.size, "dim", store.dim, "embedder", embedder.model_name)

bm25 = None
try:
    from app.chatbot.rag.bm25 import BM25Store
    bm25 = BM25Store(texts, metas=metas)
    print("BM25 ready")
except Exception as e:
    print("BM25 skip:", e)

retriever = Retriever(
    embedder=embedder,
    store=store,
    bm25=bm25,
    top_k=5,
    min_score=0.15 if "tfidf" in embedder.model_name else 0.35,
    hybrid_alpha=0.7,
)
print("min_score", retriever.min_score)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

store 100 dim 1024 embedder BAAI/bge-m3
BM25 ready
min_score 0.35


In [3]:
from app.chatbot.entities import EntityExtractor

extractor = EntityExtractor(vaccines_path=VACCINES_PATH)

samples = [
    "Tôi 20 tuổi nên tiêm vaccine nào?",
    "Vaccine HPV phòng bệnh gì?",
    "Tôi đã tiêm mũi 1 MMR rồi, tiếp theo?",
    "Khám sàng lọc trước tiêm trẻ em",
    "Chống chỉ định tiêm chủng",
]

for q in samples:
    ent = extractor.extract(q)
    vid = ent["vaccine_ids"][0] if ent["vaccine_ids"] else None
    result = retriever.retrieve(q, vaccine_id=vid)
    print("\n===" , q)
    print("entities:", {k: ent[k] for k in ("vaccine_ids", "age_years", "dose_number")})
    print("enough_evidence:", result["enough_evidence"], "max_score:", round(result["max_score"], 3))
    for h in result["hits"][:3]:
        snip = (h.get("text") or "")[:100].replace("\n", " ")
        print(f"  [{h.get('score',0):.3f}] {snip}...")


=== Tôi 20 tuổi nên tiêm vaccine nào?
entities: {'vaccine_ids': [], 'age_years': 20.0, 'dose_number': None}
enough_evidence: True max_score: 0.7
  [0.700] n thiên phổ biến (CVID) - Suy giảm miễn dịch không có Gammaglobulin - Vắc xin sốt vàng - Vắc xin bại...
  [0.614] n Haemophilus influenzae týp b đơn giá hoặc vắc xin phối hợp có chứa thành phần Haemophilus influenz...

=== Vaccine HPV phòng bệnh gì?
entities: {'vaccine_ids': [11], 'age_years': None, 'dose_number': None}
enough_evidence: True max_score: 1.0
  [1.000] Vaccine: Vắc xin HPV (Human Papillomavirus) Nhà sản xuất: MSD – Hoa Kỳ Phòng bệnh: Ung thư cổ tử cun...
  [0.368] Phác đồ: Phác đồ HPV 3 liều Vaccine: Vắc xin HPV (Human Papillomavirus) Mô tả: Phác đồ 3 liều cho nh...
  [0.360] Phác đồ: Phác đồ HPV 2 liều Vaccine: Vắc xin HPV (Human Papillomavirus) Mô tả: Phác đồ 2 liều cho nh...

=== Tôi đã tiêm mũi 1 MMR rồi, tiếp theo?
entities: {'vaccine_ids': [6], 'age_years': None, 'dose_number': 1}
enough_evidence: True max_score:

## Eval retrieval (nhanh)

Bộ query thủ công: kỳ vọng doc_id chứa keyword. Đo recall@k thô.

In [4]:
# gold: query → list substring kỳ vọng trong doc_id hoặc text
GOLD = [
    {"query": "Khám sàng lọc trước tiêm chủng", "must_contain": ["sàng lọc", "1575"]},
    {"query": "Chống chỉ định tiêm", "must_contain": ["chống chỉ định", "sàng lọc"]},
    {"query": "Phác đồ vaccine từ database", "must_contain": ["db-protocol", "db-vaccine", "protocol"]},
]

def hit_matches(h, keys):
    blob = (str(h.get("doc_id", "")) + " " + str(h.get("text", ""))).lower()
    return any(k.lower() in blob for k in keys)

k = 5
hits_at_k = 0
for g in GOLD:
    r = retriever.retrieve(g["query"], top_k=k, min_score=0.0)
    ok = any(hit_matches(h, g["must_contain"]) for h in r["hits"])
    hits_at_k += int(ok)
    print(g["query"][:40], "→", "OK" if ok else "MISS", "scores", [round(h["score"], 3) for h in r["hits"][:3]])

print(f"\nRecall@{k} (proxy) = {hits_at_k}/{len(GOLD)} = {hits_at_k/len(GOLD):.2f}")

Khám sàng lọc trước tiêm chủng → OK scores [0.977, 0.739, 0.657]
Chống chỉ định tiêm → OK scores [0.964, 0.526, 0.3]
Phác đồ vaccine từ database → OK scores [1.0, 0.649, 0.231]

Recall@5 (proxy) = 3/3 = 1.00


In [5]:
# Ghi eval thô
eval_dir = PROJECT_ROOT / "eval" / "reports"
eval_dir.mkdir(parents=True, exist_ok=True)
report = {
    "n_gold": len(GOLD),
    "hits": hits_at_k,
    "recall_at_k": hits_at_k / len(GOLD),
    "k": k,
    "embedder": embedder.model_name,
    "min_score": retriever.min_score,
    "bm25": bm25 is not None,
}
out = eval_dir / "retrieval_smoke.json"
out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("wrote", out)

wrote d:\TTTN-VaxCare-2026\vaxcare-ai\eval\reports\retrieval_smoke.json
